In [1]:
import os

In [2]:
%pwd

'c:\\Users\\DELL\\Documents\\Complete-ml-project\\Complete-ml-deployment\\research'

In [3]:
os.chdir("c:/Users/DELL/Documents/Complete-ml-project/Complete-ml-deployment")

In [4]:
%pwd

'c:\\Users\\DELL\\Documents\\Complete-ml-project\\Complete-ml-deployment'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path


In [6]:
from mlproject.constants import *
from mlproject.utils.common import read_yaml, create_directories



In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

In [8]:
from mlproject.constants import *
from mlproject.utils.common import read_yaml, create_directories

In [9]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])  


    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir),
        )

        return data_ingestion_config

In [10]:
import os 
import urllib.request as request
import zipfile
from mlproject import logger
from mlproject.utils.common import get_size

In [11]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_data(self) -> Path:
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file,
            )
            logger.info(
                f"File: {filename} downloaded with following info: \n{headers}"
            )
        else:
            logger.info(
                f"File already exists of size: {get_size(Path(self.config.local_data_file))}"
            )


    def extract_zip_file(self): 
        """
        Zip_file_path : str
        Extract the zip file into the unzip_dir
        Function does not return any thing
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"File extracted to {unzip_path}")

In [12]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_data()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2025-10-19 16:18:22,798: INFO: common]: yaml file: config\config.yaml loaded successfully
[2025-10-19 16:18:22,802: INFO: common]: yaml file: params.yaml loaded successfully
[2025-10-19 16:18:22,810: INFO: common]: yaml file: schema.yaml loaded successfully
[2025-10-19 16:18:22,817: INFO: common]: created directory at: artifacts
[2025-10-19 16:18:22,820: INFO: common]: created directory at: artifacts/data_ingestion
[2025-10-19 16:18:26,772: INFO: 3632601875]: File: artifacts\data_ingestion\data.zip downloaded with following info: 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: D7F5:1B8180:10BB31F:130AB63:68F4C901
Accept-Ranges: bytes
Dat